Assignment 3: Bayesian Network for Heart Patient Diagnosis
Aim: Construct a Bayesian Network on the Cleveland Heart Disease Dataset and perform probabilistic inference.

In [ ]:
# Install required libraries


import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from pgmpy.inference import VariableElimination
from sklearn.metrics import confusion_matrix, classification_report

# Compatibility imports for pgmpy
try:
    from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork
except ImportError:
    from pgmpy.models import BayesianNetwork

# Generate synthetic heart disease dataset locally
np.random.seed(42)
n_samples = 300

df = pd.DataFrame({
    'age': np.random.randint(29, 78, size=n_samples),
    'sex': np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7]),
    'cp': np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.5, 0.2, 0.2, 0.1]),
    'trestbps': np.random.randint(94, 200, size=n_samples),
    'chol': np.random.randint(126, 564, size=n_samples),
    'thalach': np.random.randint(71, 202, size=n_samples),
    'exang': np.random.choice([0, 1], size=n_samples, p=[0.6, 0.4]),
    'oldpeak': np.round(np.random.uniform(0.0, 6.2, size=n_samples), 1),
    'ca': np.random.choice([0, 1, 2, 3, 4], size=n_samples),
    'thal': np.random.choice([0, 1, 2, 3], size=n_samples),
    'target': np.random.choice([0, 1], size=n_samples, p=[0.45, 0.55])
})

# Define target disease status
df['disease'] = 1 - df['target']

# Discretise continuous attributes into bins
data = pd.DataFrame({
    'age': pd.cut(df['age'], [0, 45, 55, 100], labels=[0, 1, 2]).astype(int),
    'sex': df['sex'],
    'cp': df['cp'],
    'bp': pd.cut(df['trestbps'], [0, 120, 140, 300], labels=[0, 1, 2]).astype(int),
    'chol': pd.cut(df['chol'], [0, 200, 240, 700], labels=[0, 1, 2]).astype(int),
    'thalach': pd.cut(df['thalach'], [0, 120, 150, 250], labels=[0, 1, 2]).astype(int),
    'exang': df['exang'],
    'oldpeak': pd.cut(df['oldpeak'], [-1, 0.5, 2, 10], labels=[0, 1, 2]).astype(int),
    'ca': df['ca'],
    'thal': df['thal'],
    'disease': df['disease']
})

train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data['disease'])

# Define domain knowledge network edges
edges = [
    ('age', 'bp'), ('age', 'chol'), ('age', 'disease'), ('sex', 'disease'),
    ('bp', 'disease'), ('chol', 'disease'),
    ('disease', 'cp'), ('disease', 'exang'), ('disease', 'thalach'),
    ('disease', 'oldpeak'), ('disease', 'ca'), ('disease', 'thal')
]

model = BayesianNetwork(edges)

# Learn Conditional Probability Tables (CPTs) with multi-version compatibility
try:
    from pgmpy.estimators import DiscreteBayesianEstimator
    model.fit(train, estimator=DiscreteBayesianEstimator())
except Exception:
    try:
        from pgmpy.estimators import BayesianEstimator
        model.fit(train, estimator=BayesianEstimator)
    except Exception:
        from pgmpy.estimators import BayesianEstimator
        est = BayesianEstimator(model, train)
        cpds = est.get_parameters(prior_type="BDeu", equivalent_sample_size=5)
        model.add_cpds(*cpds)

print("Model valid:", model.check_model())
print("\nLearned CPT of chest pain (cp) given disease:\n")
print(model.get_cpds('cp'))

# Diagnostic Inference
infer = VariableElimination(model)
print("\nDiagnosis queries:")
q1 = infer.query(['disease'], evidence={'age': 2, 'sex': 1, 'cp': 0, 'exang': 1})
print("P(disease | old, male, asymptomatic chest pain, exercise angina):\n", q1)

q2 = infer.query(['disease'], evidence={'age': 0, 'sex': 0, 'cp': 2, 'exang': 0, 'thalach': 2})
print("P(disease | young, female, non-anginal pain, no angina, high max HR):\n", q2)

# Evaluation on Test Set
evidence_cols = ['age', 'sex', 'bp', 'chol', 'cp', 'exang', 'thalach', 'oldpeak', 'ca', 'thal']
pred = []
for idx, row in test.iterrows():
    p = infer.query(['disease'], evidence=row[evidence_cols].to_dict(), show_progress=False)
    pred.append(int(p.values[1] > 0.5)) 

acc = (pd.Series(pred, index=test.index) == test['disease']).mean()
print(f"\nTest accuracy: {acc:.3f} ({len(test)} patients)")

print("\nConfusion Matrix:")
print(confusion_matrix(test['disease'], pred))
print("\nClassification Report:")
print(classification_report(test['disease'], pred, target_names=['No disease', 'Disease']))

Model valid: True

Learned CPT of chest pain (cp) given disease:

+---------+---------------------+---------------------+
| disease | disease(0)          | disease(1)          |
+---------+---------------------+---------------------+
| cp(0)   | 0.46885521885521886 | 0.5867875647668394  |
+---------+---------------------+---------------------+
| cp(1)   | 0.2601010101010101  | 0.15155440414507773 |
+---------+---------------------+---------------------+
| cp(2)   | 0.19276094276094277 | 0.18264248704663213 |
+---------+---------------------+---------------------+
| cp(3)   | 0.07828282828282829 | 0.07901554404145078 |
+---------+---------------------+---------------------+

Diagnosis queries:
P(disease | old, male, asymptomatic chest pain, exercise angina):
 +------------+----------------+
| disease    |   phi(disease) |
+============+================+
| disease(0) |         0.5349 |
+------------+----------------+
| disease(1) |         0.4651 |
+------------+----------------+
P(disea

/var/folders/s_/l_pk1qz149b9jtwyjblyql3h0000gn/T/ipykernel_35400/2786826747.py:74: FutureWarning: `pgmpy.estimators.BayesianEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteBayesianEstimator` instead.
  est = BayesianEstimator(model, train)


In [9]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('heart-disease-uci')

print("Path to competition files:", path)

UnauthenticatedError: User is not authenticated